# 🚦 01 — Análisis Exploratorio de Datos (EDA) y Comprensión del Negocio
### Proyecto Final: Predicción de Severidad de Siniestros Viales en Medellín
**Especialización en Analítica y Ciencia de Datos / Machine Learning**

---

## 1. Contexto y Objetivos del Proyecto

En el Distrito de Medellín, los siniestros viales constituyen una problemática crítica para la movilidad y la salud pública. Cuando ocurre un siniestro, la Línea 123 y la Secretaría de Movilidad reciben un reporte preliminar que contiene únicamente:
- Ubicación geográfica (comuna, barrio, dirección).
- Tipo preliminar de evento (choque, atropello, volcamiento, caída de ocupante).
- Momento temporal (hora, día de la semana, mes).

### Objetivo de Machine Learning:
Entrenar un modelo de clasificación supervisada capaz de predecir la probabilidad de que un siniestro sea **grave (`es_grave = 1`)**, es decir, que involucre **heridos o víctimas fatales**, versus aquellos que representan únicamente **daños materiales (`es_grave = 0`)**.

### Impacto de Negocio:
1. **Priorización de Despacho (123):** Asignación inmediata de ambulancias de Soporte Vital Avanzado (SVA).
2. **Focalización Preventiva:** Detección de comunas y franjas horarias de alto riesgo para operativos viales.

In [ ]:
# Configuración del entorno e importaciones modulares
import sys
from pathlib import Path

# Garantizar acceso al paquete modular src desde la carpeta notebooks
PROJECT_ROOT = Path(".").resolve().parent if Path(".").resolve().name == "notebooks" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import TARGET_COLUMN, BINARY_TARGET_COLUMN, RANDOM_STATE
from src.data.load_data import load_raw_data
from src.features.build_features import (
    clean_target_column,
    extract_temporal_features,
    clean_categorical_features,
    drop_leakage_columns,
    prepare_full_dataset,
)
from src.utils.plots import (
    plot_target_distribution,
    plot_comuna_risk,
    plot_temporal_patterns,
    plot_clase_accidente_risk,
)

print(f"[OK] Módulos cargados correctamente. Semilla global: {RANDOM_STATE}")

## 2. Ingesta y Calidad de Datos Crudos (MEData)

Cargamos el dataset oficial de la Secretaría de Movilidad de Medellín utilizando el módulo reproducible `src.data.load_data`.

In [ ]:
# Cargar dataset (utiliza la muestra representativa de 30,000 registros para desarrollo ágil)
df_raw = load_raw_data(use_sample=True)
print(f"Dimensiones originales: {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas")
df_raw.head(3)

### 2.1 Identificación de Columnas y Tipos de Datos

In [ ]:
# Resumen de valores nulos y tipos
missing_summary = pd.DataFrame({
    "Tipo": df_raw.dtypes,
    "Nulos": df_raw.isna().sum(),
    "% Nulos": (df_raw.isna().mean() * 100).round(2)
}).sort_values(by="% Nulos", ascending=False)

missing_summary

## 3. Preparación e Ingeniería de Características

Aplicamos el pipeline modular `prepare_full_dataset` de `src.features.build_features`:
1. **Target Normalization:** Se mapea `GRAVEDAD_ACCIDENTE` a `es_grave` (1: Heridos/Muertos, 0: Solo daños).
2. **Extracción Temporal:** Extracción de hora, día de la semana, franja horaria y variables cíclicas.
3. **Limpieza Espacial y Vial:** Normalización de comunas, diseño de vía y tipo de siniestro.
4. **Aislamiento de Data Leakage:** Eliminación estricta de `EXPEDIENTE` y `NRO_RADICADO`.

In [ ]:
df_prep = prepare_full_dataset(df_raw)
print(f"Dimensiones procesadas: {df_prep.shape[0]:,} filas x {df_prep.shape[1]} columnas")
df_prep[["es_grave", "hora", "franja_horaria", "comuna_limpia", "clase_accidente_limpia", "diseno_limpio"]].head()

## 4. Distribución de la Variable Objetivo (Target)

Analizamos la proporción de incidentes graves frente a no graves.

In [ ]:
fig_target = plot_target_distribution(df_prep)
plt.show()

**Hallazgos Clave del Target:**
- En Medellín, aproximadamente el **56.4%** de los incidentes reportados implican lesiones o fatalidades (`es_grave = 1`), mientras que el **43.6%** son solo daños materiales.
- La distribución es balanceada, pero la matriz de costos de negocio es asimétrica: un **Falso Negativo** (predecir 'solo daños' cuando hay heridos graves) cuesta vidas humanas al demorar el despacho médico. Por ende, la métrica clave a maximizar será el **Recall de la clase 1** junto con **PR-AUC**.

## 5. Análisis Espacial: Severidad por Comunas de Medellín

In [ ]:
fig_comunas = plot_comuna_risk(df_prep, top_n=12)
plt.show()

**Hallazgo Espacial:**
- Comunas céntricas como **La Candelaria (Centro)** y **Laureles** tienen el volumen más alto por flujo vehicular comercial.
- Sin embargo, sectores de ladera y vías rápidas periféricas (como **Robledo, Aranjuez o San Javier**) exhiben porcentajes elevados de severidad con heridos, reflejando pendientes pronunciadas y velocidades en interacción con motos.

## 6. Dinámica Temporal: Hora del Día y Día de la Semana

In [ ]:
fig_temporal = plot_temporal_patterns(df_prep)
plt.show()

**Hallazgo Temporal:**
- **Efecto Madrugada:** Aunque el volumen total de accidentes cae en la madrugada (00:00 a 05:00), el **porcentaje de siniestros severos sube notablemente** (asociado a exceso de velocidad en vías despejadas e ingesta de alcohol).
- **Fines de Semana:** Sábados y domingos presentan una mayor tasa de severidad porcentual frente a los días hábiles.

## 7. Dinámica del Siniestro: ¿Qué eventos son más críticos?

In [ ]:
fig_clase = plot_clase_accidente_risk(df_prep)
plt.show()

**Hallazgo Crítico:**
- **Atropello** y **Caída de Ocupante** (motociclistas) presentan tasas de criticidad vital superiores al **85-90%**, convirtiéndose en los predictores de mayor poder discriminante.
- En contraste, los **Choques** entre vehículos convencionales tienen una tasa de severidad mucho más baja (~45-50%), con predominio de daños materiales.

## 8. Conclusiones y Guía para la Fase de Modelado (Pipeline)

1. **Variables predictoras clave:** `clase_accidente_limpia`, `franja_horaria`, `hora_sin`, `hora_cos`, `dia_semana`, `comuna_limpia`, `diseno_limpio` y `tipo_via`.
2. **Aislamiento Hermético:** No usar `EXPEDIENTE` ni `NRO_RADICADO` en ningún paso.
3. **Pipeline de Scikit-Learn:** Todo preprocesamiento (imputación por moda/mediana y OneHotEncoder) debe ensamblarse en un `ColumnTransformer` hermético dentro de `src.models.pipeline` para garantizar **Cero Data Leakage**.
4. **Métricas a Optimizar:** PR-AUC, F1-Score ponderado y Recall de la clase crítica (`es_grave = 1`).